In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/goemotions_2.csv')
df.head()

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,We can hope,ee3o3ko,darkenseyreth,EdmontonOilers,t3_ag4r9j,t1_ee3mhad,1.547529e+09,62,False,0,...,0,0,1,0,0,0,0,0,0,0
1,Shhh don't give them the idea!,eebl3z7,BoinkBoinkEtAliae,MurderedByWords,t3_ah3o76,t1_eeb68lo,1.547777e+09,76,False,0,...,0,0,0,0,0,0,0,0,0,0
2,"Thank you so much, kind stranger. I really nee...",ed4fe9l,savageleaf,raisedbynarcissists,t3_abwh00,t1_ed4etbj,1.546482e+09,24,False,0,...,0,0,0,0,0,0,0,0,0,0
3,Ion know but it would be better for you to jus...,efavtdu,CADDiLLXC,darknet,t3_al4njw,t3_al4njw,1.548800e+09,62,False,0,...,0,0,0,0,0,0,0,0,0,1
4,I'm honestly surprised. We should have fallen ...,ee2imz2,CorporalThornberry,CollegeBasketball,t3_afxt6t,t1_ee22nyr,1.547497e+09,55,False,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
emotion_columns = [
'admiration','amusement','anger','annoyance','approval','caring',
'confusion','curiosity','desire','disappointment','disapproval',
'disgust','embarrassment','excitement','fear','gratitude','grief',
'joy','love','nervousness','optimism','pride','realization','relief',
'remorse','sadness','surprise','neutral'
]

In [ ]:
df['emotion'] = df[emotion_columns].idxmax(axis=1)

In [ ]:
selected_emotions = [
'amusement',
'gratitude',
'anger',
'disappointment',
'confusion',
'love',
'joy',
'sadness',
'surprise',
'fear'
]

df_filtered = df[df['emotion'].isin(selected_emotions)]

In [ ]:
emotion_to_polarity = {
    'joy': 'positive',
    'love': 'positive',
    'gratitude': 'positive',
    'amusement': 'positive',

    'anger': 'negative',
    'sadness': 'negative',
    'fear': 'negative',
    'disappointment': 'negative',

    'confusion': 'neutral',
    'surprise': 'neutral'
}

In [ ]:
df_balanced = df_filtered.groupby('emotion').sample(
    n=3000,
    replace=True,
    random_state=42
)

In [ ]:
X = df_balanced['text']
y = df_balanced['emotion']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    stop_words='english'
)
X_vectorized = vectorizer.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=4000,
    solver='saga',
    class_weight='balanced',
    n_jobs=-1
)
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=4000, n_jobs=-1,
                   solver='saga')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.7551666666666667

Classification Report:

                precision    recall  f1-score   support

     amusement       0.80      0.73      0.76       600
         anger       0.71      0.71      0.71       600
     confusion       0.65      0.69      0.67       600
disappointment       0.60      0.59      0.60       600
          fear       0.86      0.91      0.88       600
     gratitude       0.86      0.82      0.84       600
           joy       0.76      0.66      0.71       600
          love       0.81      0.85      0.83       600
       sadness       0.76      0.75      0.75       600
      surprise       0.76      0.85      0.80       600

      accuracy                           0.76      6000
     macro avg       0.76      0.76      0.75      6000
  weighted avg       0.76      0.76      0.75      6000



In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_selection import SelectKBest, chi2

df = pd.read_csv('/content/drive/MyDrive/goemotions_2.csv')

emotion_columns = [
    'admiration','amusement','anger','annoyance','approval','caring',
    'confusion','curiosity','desire','disappointment','disapproval',
    'disgust','embarrassment','excitement','fear','gratitude','grief',
    'joy','love','nervousness','optimism','pride','realization','relief',
    'remorse','sadness','surprise','neutral'
]

df['emotion'] = df[emotion_columns].idxmax(axis=1)

selected_emotions = [
    'amusement', 'gratitude', 'anger', 'disappointment',
    'confusion', 'love', 'joy', 'sadness', 'surprise', 'fear'
]

df_filtered = df[df['emotion'].isin(selected_emotions)]

target_samples = 3000
df_balanced_list = []

for emotion in selected_emotions:
    subset = df_filtered[df_filtered['emotion'] == emotion]

    if len(subset) >= target_samples:
        sampled = subset.sample(n=target_samples, random_state=42)
    else:
        sampled = subset.sample(n=target_samples, replace=True, random_state=42)

    df_balanced_list.append(sampled)

df_balanced = pd.concat(df_balanced_list).reset_index(drop=True)

print("Balanced dataset size:", df_balanced.shape)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_balanced['text'] = df_balanced['text'].apply(clean_text)

X = df_balanced['text']
y = df_balanced['emotion']

vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    stop_words='english'
)

X_vectorized = vectorizer.fit_transform(X)

selector = SelectKBest(chi2, k=20000)
X_vectorized = selector.fit_transform(X_vectorized, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LinearSVC(
    C=2.5,
    class_weight='balanced',
    max_iter=12000
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=3))

cv_scores = cross_val_score(model, X_vectorized, y, cv=5)

print("\nCross-validation scores:", cv_scores)
print("Average CV accuracy:", cv_scores.mean())

emotion_to_polarity = {
    'joy': 'positive',
    'love': 'positive',
    'gratitude': 'positive',
    'amusement': 'positive',
    'anger': 'negative',
    'sadness': 'negative',
    'fear': 'negative',
    'disappointment': 'negative',
    'confusion': 'neutral',
    'surprise': 'neutral'
}

def predict_emotion_and_polarity(text):
    text = clean_text(text)
    text_vector = vectorizer.transform([text])
    text_vector = selector.transform(text_vector)

    emotion = model.predict(text_vector)[0]
    polarity = emotion_to_polarity.get(emotion, "neutral")

    return emotion, polarity

print("\nSample Prediction:")
print(predict_emotion_and_polarity("I am extremely happy today"))

Balanced dataset size: (30000, 38)

Test Accuracy: 0.835

Classification Report:

                precision    recall  f1-score   support

     amusement      0.845     0.812     0.828       600
         anger      0.776     0.768     0.772       600
     confusion      0.771     0.782     0.776       600
disappointment      0.709     0.705     0.707       600
          fear      0.919     0.940     0.929       600
     gratitude      0.917     0.883     0.900       600
           joy      0.827     0.813     0.820       600
          love      0.863     0.900     0.881       600
       sadness      0.859     0.852     0.855       600
      surprise      0.863     0.895     0.879       600

      accuracy                          0.835      6000
     macro avg      0.835     0.835     0.835      6000
  weighted avg      0.835     0.835     0.835      6000


Cross-validation scores: [0.82066667 0.826      0.83066667 0.83033333 0.83333333]
Average CV accuracy: 0.8282

Sample Prediction:


In [ ]:

!pip install -q sentence-transformers xgboost lightgbm scikit-learn

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.utils import resample
from xgboost import XGBClassifier
import lightgbm as lgb

df = pd.read_csv('/content/drive/MyDrive/goemotions_2.csv')

emotion_columns = [
    'admiration','amusement','anger','annoyance','approval','caring',
    'confusion','curiosity','desire','disappointment','disapproval',
    'disgust','embarrassment','excitement','fear','gratitude','grief',
    'joy','love','nervousness','optimism','pride','realization','relief',
    'remorse','sadness','surprise','neutral'
]

df['label_count'] = df[emotion_columns].sum(axis=1)
df = df[df['label_count'] == 1]
df['emotion'] = df[emotion_columns].idxmax(axis=1)

selected_emotions = ['joy','anger','fear','sadness','surprise','gratitude']
df = df[df['emotion'].isin(selected_emotions)]

df_list = []
for label in selected_emotions:
    class_df = df[df['emotion']==label]
    if label in ['surprise','gratitude']:
        class_df = resample(class_df, replace=True, n_samples=1200, random_state=42)
    else:
        class_df = resample(class_df, replace=True, n_samples=1000, random_state=42)
    df_list.append(class_df)
df_balanced = pd.concat(df_list)

le = LabelEncoder()
df_balanced['label'] = le.fit_transform(df_balanced['emotion'])
y = df_balanced['label'].values
id2label = {i: label for i, label in enumerate(le.classes_)}

model_mini = SentenceTransformer('all-MiniLM-L6-v2')
model_mpnet = SentenceTransformer('all-mpnet-base-v2')

def get_embeddings(texts):
    emb_mini = model_mini.encode(texts, batch_size=64, show_progress_bar=True)
    emb_mpnet = model_mpnet.encode(texts, batch_size=64, show_progress_bar=True)
    emb_combined = np.concatenate([emb_mini, emb_mpnet], axis=1)
    return emb_mini, emb_mpnet, emb_combined

X_mini, X_mpnet, X_combined = get_embeddings(df_balanced['text'].tolist())

scaler_mini = StandardScaler()
scaler_mpnet = StandardScaler()
scaler_combined = StandardScaler()

X_mini_scaled = scaler_mini.fit_transform(X_mini)
X_mpnet_scaled = scaler_mpnet.fit_transform(X_mpnet)
X_combined_scaled = scaler_combined.fit_transform(X_combined)

X_train_mini, X_test_mini, y_train, y_test = train_test_split(
    X_mini_scaled, y, test_size=0.2, stratify=y, random_state=42
)
X_train_mpnet, X_test_mpnet, _, _ = train_test_split(
    X_mpnet_scaled, y, test_size=0.2, stratify=y, random_state=42
)
X_train_comb, X_test_comb, _, _ = train_test_split(
    X_combined_scaled, y, test_size=0.2, stratify=y, random_state=42
)


X_train_comb_df = pd.DataFrame(X_train_comb, columns=[f"f{i}" for i in range(X_train_comb.shape[1])])
X_test_comb_df = pd.DataFrame(X_test_comb, columns=[f"f{i}" for i in range(X_test_comb.shape[1])])

model_xgb_mini = XGBClassifier(n_estimators=700, max_depth=6, learning_rate=0.08, eval_metric='mlogloss')
model_xgb_mini.fit(X_train_mini, y_train)

model_xgb_mpnet = XGBClassifier(n_estimators=700, max_depth=6, learning_rate=0.08, eval_metric='mlogloss')
model_xgb_mpnet.fit(X_train_mpnet, y_train)

model_lgb = lgb.LGBMClassifier(n_estimators=700, max_depth=6, learning_rate=0.08)
model_lgb.fit(X_train_comb_df, y_train)

def ensemble_predict_test(X_mini_scaled, X_mpnet_scaled, X_comb_scaled_df):
    prob1 = model_xgb_mini.predict_proba(X_mini_scaled)
    prob2 = model_xgb_mpnet.predict_proba(X_mpnet_scaled)
    prob3 = model_lgb.predict_proba(X_comb_scaled_df)
    avg_prob = (prob1 + prob2 + prob3) / 3
    preds = np.argmax(avg_prob, axis=1)
    return preds

def get_polarity(emotion):
    mapping = {
        'joy': 'positive',
        'gratitude': 'positive',
        'surprise': 'neutral',
        'anger': 'negative',
        'sadness': 'negative',
        'fear': 'negative'
    }
    return mapping[emotion]

y_pred_test = ensemble_predict_test(X_test_mini, X_test_mpnet, X_test_comb_df)
y_pred_emotions = [id2label[i] for i in y_pred_test]
y_true_emotions = [id2label[i] for i in y_test]

emotion_accuracy = accuracy_score(y_true_emotions, y_pred_emotions)
print(f" Overall Emotion Accuracy: {emotion_accuracy*100:.2f}%")

y_pred_polarity = [get_polarity(e) for e in y_pred_emotions]
y_true_polarity = [get_polarity(e) for e in y_true_emotions]
polarity_accuracy = accuracy_score(y_true_polarity, y_pred_polarity)
print(f" Overall Polarity Accuracy: {polarity_accuracy*100:.2f}%")

def ensemble_predict(texts):
    emb_mini, emb_mpnet, emb_comb = get_embeddings(texts)

    mini_scaled = scaler_mini.transform(emb_mini)
    mpnet_scaled = scaler_mpnet.transform(emb_mpnet)
    comb_scaled_df = pd.DataFrame(scaler_combined.transform(emb_comb), columns=[f"f{i}" for i in range(emb_comb.shape[1])])

    prob1 = model_xgb_mini.predict_proba(mini_scaled)
    prob2 = model_xgb_mpnet.predict_proba(mpnet_scaled)
    prob3 = model_lgb.predict_proba(comb_scaled_df)

    avg_prob = (prob1 + prob2 + prob3) / 3
    preds = np.argmax(avg_prob, axis=1)
    emotions = [id2label[i] for i in preds]
    polarities = [get_polarity(e) for e in emotions]
    return emotions, polarities

test_texts = ["I feel amazing today!", "Wow, this is surprising!"]
pred_emotions, pred_polarities = ensemble_predict(test_texts)

for t, e, p in zip(test_texts, pred_emotions, pred_polarities):
    print(f"\nText: {t}\n Emotion: {e}\n Polarity: {p}")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.116745 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 293760
[LightGBM] [Info] Number of data points in the train set: 5120, number of used features: 1152
[LightGBM] [Info] Start training from score -1.856298
[LightGBM] [Info] Start training from score -1.856298
[LightGBM] [Info] Start training from score -1.673976
[LightGBM] [Info] Start training from score -1.856298
[LightGBM] [Info] Start training from score -1.856298
[LightGBM] [Info] Start training from score -1.673976
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Text: I feel amazing today!
 Emotion: joy
 Polarity: positive

Text: Wow, this is surprising!
 Emotion: surprise
 Polarity: neutral


In [ ]:

!pip install -q sentence-transformers xgboost lightgbm scikit-learn


import pandas as pd
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

from xgboost import XGBClassifier
import lightgbm as lgb



def handle_negation(text):
    text = text.lower()
    text = re.sub(r'\b(not|no|never)\s+(\w+)', r'not_\2', text)
    text = re.sub(r'\bun(\w+)', r'not_\1', text)
    return text

df = pd.read_csv('/content/drive/MyDrive/goemotions_2.csv')

emotion_columns = [
    'admiration','amusement','anger','annoyance','approval','caring',
    'confusion','curiosity','desire','disappointment','disapproval',
    'disgust','embarrassment','excitement','fear','gratitude','grief',
    'joy','love','nervousness','optimism','pride','realization','relief',
    'remorse','sadness','surprise','neutral'
]

df['label_count'] = df[emotion_columns].sum(axis=1)
df = df[df['label_count'] == 1]
df['emotion'] = df[emotion_columns].idxmax(axis=1)

selected_emotions = ['joy','anger','fear','sadness','surprise','gratitude']
df = df[df['emotion'].isin(selected_emotions)]

df['text'] = df['text'].apply(handle_negation)

df_list = []
for label in selected_emotions:
    class_df = df[df['emotion'] == label]
    class_df = resample(class_df, replace=True, n_samples=1500, random_state=42)
    df_list.append(class_df)

df_balanced = pd.concat(df_list)

le = LabelEncoder()
df_balanced['label'] = le.fit_transform(df_balanced['emotion'])
y = df_balanced['label'].values
id2label = {i: label for i, label in enumerate(le.classes_)}

model_mini = SentenceTransformer('all-MiniLM-L6-v2')
model_mpnet = SentenceTransformer('all-mpnet-base-v2')
model_roberta = SentenceTransformer('all-distilroberta-v1')

def get_embeddings(texts):
    texts = [handle_negation(t) for t in texts]  # apply during prediction also

    emb1 = model_mini.encode(texts, batch_size=64)
    emb2 = model_mpnet.encode(texts, batch_size=64)
    emb3 = model_roberta.encode(texts, batch_size=64)

    combined = np.concatenate([emb1, emb2, emb3], axis=1)
    return emb1, emb2, emb3, combined


X1, X2, X3, X_comb = get_embeddings(df_balanced['text'].tolist())

sc1, sc2, sc3, sc_comb = StandardScaler(), StandardScaler(), StandardScaler(), StandardScaler()

X1 = sc1.fit_transform(X1)
X2 = sc2.fit_transform(X2)
X3 = sc3.fit_transform(X3)
X_comb = sc_comb.fit_transform(X_comb)

X1_tr, X1_te, y_tr, y_te = train_test_split(X1, y, test_size=0.2, stratify=y, random_state=42)
X2_tr, X2_te, _, _ = train_test_split(X2, y, test_size=0.2, stratify=y, random_state=42)
X3_tr, X3_te, _, _ = train_test_split(X3, y, test_size=0.2, stratify=y, random_state=42)
Xc_tr, Xc_te, _, _ = train_test_split(X_comb, y, test_size=0.2, stratify=y, random_state=42)

Xc_tr_df = pd.DataFrame(Xc_tr)
Xc_te_df = pd.DataFrame(Xc_te)

xgb1 = XGBClassifier(n_estimators=1200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
xgb2 = XGBClassifier(n_estimators=1200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')
xgb3 = XGBClassifier(n_estimators=1200, max_depth=6, learning_rate=0.05, eval_metric='mlogloss')

lgbm = lgb.LGBMClassifier(n_estimators=1200, learning_rate=0.05)

xgb1.fit(X1_tr, y_tr)
xgb2.fit(X2_tr, y_tr)
xgb3.fit(X3_tr, y_tr)
lgbm.fit(Xc_tr_df, y_tr)

def predict_ensemble(X1, X2, X3, Xc):
    p1 = xgb1.predict_proba(X1)
    p2 = xgb2.predict_proba(X2)
    p3 = xgb3.predict_proba(X3)
    p4 = lgbm.predict_proba(Xc)

    final = (0.2*p1 + 0.3*p2 + 0.3*p3 + 0.2*p4)
    return np.argmax(final, axis=1)

y_pred = predict_ensemble(X1_te, X2_te, X3_te, Xc_te_df)

y_pred_labels = [id2label[i] for i in y_pred]
y_true_labels = [id2label[i] for i in y_te]

print(f"\n FINAL Accuracy: {accuracy_score(y_true_labels, y_pred_labels):.4f}\n")
print(classification_report(y_true_labels, y_pred_labels, digits=2))

def get_polarity(e):
    return {
        'joy':'positive','gratitude':'positive',
        'surprise':'neutral',
        'anger':'negative','sadness':'negative','fear':'negative'
    }[e]

texts = [
    "I feel amazing today!",
    "I feel unhappy today",
    "Wow, this is surprising!"
]

e1, e2, e3, ec = get_embeddings(texts)

e1 = sc1.transform(e1)
e2 = sc2.transform(e2)
e3 = sc3.transform(e3)
ec = pd.DataFrame(sc_comb.transform(ec))

preds = predict_ensemble(e1, e2, e3, ec)

for t, p in zip(texts, preds):
    emo = id2label[p]
    pol = get_polarity(emo)
    print(f"\nText: {t}\n Emotion: {emo}\n Polarity: {pol}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.360429 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 489600
[LightGBM] [Info] Number of data points in the train set: 7200, number of used features: 1920
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

In [ ]:
import joblib

# Save models
joblib.dump(xgb1, "xgb1.pkl")
joblib.dump(xgb2, "xgb2.pkl")
joblib.dump(xgb3, "xgb3.pkl")
joblib.dump(lgbm, "lgbm.pkl")

# Save scalers
joblib.dump(sc1, "sc1.pkl")
joblib.dump(sc2, "sc2.pkl")
joblib.dump(sc3, "sc3.pkl")
joblib.dump(sc_comb, "sc_comb.pkl")

# Save label mapping/
joblib.dump(id2label, "id2label.pkl")

print(" All models saved successfully!")

In [ ]:
import shutil

files = [
    "xgb1.pkl","xgb2.pkl","xgb3.pkl","lgbm.pkl",
    "sc1.pkl","sc2.pkl","sc3.pkl","sc_comb.pkl",
    "id2label.pkl"
]

for f in files:
    shutil.move(f, "/content/drive/MyDrive/" + f)

print(" Files moved to Google Drive!")

In [ ]:
import joblib

# Save models
joblib.dump(xgb1, "xgb1.pkl")
joblib.dump(xgb2, "xgb2.pkl")
joblib.dump(xgb3, "xgb3.pkl")
joblib.dump(lgbm, "lgbm.pkl")

# Save scalers
joblib.dump(sc1, "sc1.pkl")
joblib.dump(sc2, "sc2.pkl")
joblib.dump(sc3, "sc3.pkl")
joblib.dump(sc_comb, "sc_comb.pkl")

# Save label mapping
joblib.dump(id2label, "id2label.pkl")

print("All models saved!")